# **Correlation Study**

## Objectives

* Expand on the cancellation EDA to answer BR1:

    *TCS Hotels wants to understand cancellation patterns, trends and guest behaviour across their 2 Portuguese properties in order to identify risk factors and develop more effective cancellation defence strategies*
* Investigate statistical relationships between features and the target variable `is_canceled` using Pearson, Spearman and PPS to indentify which features carry meaningful predictive signal
* Formally validate hypotheses 1, 2 and 3 using appropriate statistical tests
* Rank features by predictive relationship with the target to inform which engineered features are required

## Inputs

* The dataset as cleaned in the [cleaning notebook](/jupyter_notebooks/04_cleaning.ipynb):

    outputs/datasets/cleaned/HotelBookingsClean.csv

## Outputs

* Relevant correlation/PPS matrices and visualisations
* Statistical test results for H1, H2 and H3
* A ranked features table saved to outputs/correlation/RankedFeatures.csv

## Additional Comments

* p-value returned as 0.0 due to floating-point underflow at this sample size; reported as p < .001


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

## Load Data

* Load the fully cleaned dataset for correlation analysis

In [ ]:
import pandas as pd
df = pd.read_csv("outputs/datasets/cleaned/HotelBookingsClean.csv")
df.head(3)

In [ ]:
print(f"Dataframe shape is: {df.shape}")


* Following cleaning, the dataset has a shape of 119,187 rows across 28 features

In [ ]:
df.info()

* Upon inspection it appears that the category conversion of `is_repeated_guest`, `agent` and `company` was not preserved in the saving process.
* Re-specify dtype

In [ ]:
categorical_list = ["is_repeated_guest", "agent", "company"]
for col in categorical_list:
    df[col] = df[col].astype("category")

df.info()

* The dataset now consists of 1 float variable, 3 category, 9 object and 15 integer. 
* Excluding the target variable `is_canceled`, there are 15 numeric features and 12 categorical features
* Numeric features include 3 temporal features `arrival_date_year`, `arrival_date_week_number` and `arrival_date_day_of_month`, these are held out from the Pearson and Spearman correlation tests due to their temporal nature. `arrival_date_week_number` and `arrival_date_day_of_month` are both somewhat cyclical in nature rather than truly numeric and `arrival_date_year` is a fixed point in history that can have no bearing on current predictions
* Categorical features also include a temporal feature `arrival_date_month` which is included the temporal features rather than the categorical due to its cyclical nature
* All features are tested during pps analysis, so any predictive importance in the temporal features can surface there.

* Check the cancellation rate is unchanged following cleaning

In [ ]:
df["is_canceled"].value_counts(normalize=True)

* The overall cancellation rate is still ~37%

---

## Split Feature Groups

* The dataset contains 4 distinct feature groups:
    1. The target variable (binary) ["is_canceled"]
    2. Numeric Features
    3. Categorical Features
    4. Temporal Features

In [ ]:
target = df["is_canceled"]

In [ ]:
numeric_features = [
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "adults",
    "children",
    "babies",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "days_in_waiting_list",
    "adr",
    "required_car_parking_spaces",
    "total_of_special_requests",
]

In [ ]:
categorical_features = [
    "hotel",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type",
    "is_repeated_guest",
    "agent",
    "company",
]

In [ ]:
temporal_features = [
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month"
]

* Create dataframe of numeric features for correlation tests

In [ ]:
numeric_correlation_df = pd.concat([target, df[numeric_features]], axis=1)
numeric_correlation_df.head()

---

## Pearson Correlation with Target

In [ ]:
pearson_df = numeric_correlation_df.corr(method="pearson")
pearson_df

* Create heatmap to visualise the correlations

In [ ]:
# Create mask to remove duplicated values and below the threshold
import numpy as np

def heatmap_mask(df, threshold):
    mask = np.zeros_like(df, dtype=bool)
    mask[np.triu_indices_from(mask)] = True
    mask[abs(df) < threshold] = True
    return mask


* Values below ±0.1 are masked to improve readability because correlations below this level are generally considered negligible for exploratory analysis.

In [ ]:
threshold = 0.1

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle("Pearson correlation matrix")

sns.heatmap(
    pearson_df, 
    mask=(heatmap_mask(pearson_df, threshold)), 
    annot=True,
    cmap="winter",
    ax=ax)

plt.show()

* The Pearson heatmap shows that dataset only has weak correlations between the numeric variables with only `lead_time`, `previous_cancellations`, `required_car_parking_spaces` and `total_of_special_requests` clearing the 0.1 threshold for correlation with the target `is_canceled`

---

## Spearman Correlation with Target

In [ ]:
spearman_df = numeric_correlation_df.corr(method="spearman")
spearman_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle("Spearman correlation matrix")

sns.heatmap(
    spearman_df, 
    mask=(heatmap_mask(spearman_df, threshold)), 
    annot=True,
    cmap="winter",
    ax=ax)

plt.show()

* The Spearman correlation heatmap shows slightly stronger correlations with a few more features correlating and the addition of `previous_bookings_not_canceled` in the list of variables meeting the threshold value against target

**Compare Pearson and Spearman correlations focusing on the target variable.**

In [ ]:
pearson_is_canceled = pearson_df["is_canceled"].drop(labels="is_canceled")
spearman_is_canceled = spearman_df["is_canceled"].drop(labels="is_canceled")

comparison_df = pd.concat({"Spearman": spearman_is_canceled, "Pearson": pearson_is_canceled}, axis=1)
comparison_df["Difference"] = comparison_df["Spearman"] - comparison_df["Pearson"]

comparison_df


* The two correlation methods largely agree with `previous_cancellations` having the largest difference between the two methods at 0.16

* Prepare the data for plotting

In [ ]:
# Remove the difference column and make the feature index a true column 
plot_comparison = comparison_df.reset_index(names="Feature")
plot_comparison = plot_comparison.drop(columns="Difference")
plot_comparison

In [ ]:
# Combine the 2 method columns into method/value columns
plot_comparison = plot_comparison.melt(id_vars="Feature", value_vars=["Pearson", "Spearman"], var_name="Method", value_name="Value")
plot_comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle("Pearson vs Spearman Correlations")

ax.set_xlabel("Correlation Coefficient")

sns.barplot(data=plot_comparison,
            x="Value",
            y="Feature",
            hue="Method")

plt.show()

* From the chart we can see the divergence of `previous_cancellations` flagged by the comparison dataframe, but can also see that `previous_bookings_not_canceled` and `days_in_waiting_list` diverge, where Spearman scores are notably larger. This is consistent with their shared heavily-skewed, zero-inflated distributions. 

---

## Predictive Power Score

* To assess the categorical features, we will use ppscore

In [ ]:
import ppscore as pps

pps_df = df.copy()

pps_df["is_canceled"] = pps_df["is_canceled"].astype("category")

pps_predict = pps.predictors(pps_df, y="is_canceled", sample=None)
pps_predict.sort_values(by="ppscore", ascending=False)


* Only 7 features returned any predictive power score.
* Separate only the rows with a ppscore value

In [ ]:
pps_plot_df = pps_predict[pps_predict["ppscore"] > 0].sort_values(by="ppscore", ascending=False) 
pps_plot_df


* The 2 strongest predictive features appear to be `deposit_type` and `country` with ppscores above 0.3
* `agent`, `adr`, `lead_time` and `market_segment` all have ppscores above 0.2

In [ ]:
fig, ax = plt.subplots()
fig.suptitle("PPS against target")
ax.set_ylabel("Feature")
ax.set_xlabel("PPS Score")

sns.barplot(data=pps_plot_df,
            x="ppscore",
            y="x")

plt.show()

---

## Feature Ranking

* From the 3 correlation methods performed, we can now derive a set of relevant features
* The threshold of 0.1 has been kept for Pearson and Spearman correlations as a preferred cut-off rather than simply taking the top n features from each, due to the low overall correlation values

In [ ]:
pearson = pd.Series(pearson_df["is_canceled"]).drop(labels="is_canceled")
pearson = pearson.reindex(pearson.abs().sort_values(ascending=False).index)
pearson = pearson[pearson.abs() > threshold]
pearson

* The 4 features from the Pearson correlation are `lead_time`, `total_of_special_requests`, `required_car_parking_spaces` and `previous_cancellations`

In [ ]:
spearman = pd.Series(spearman_df["is_canceled"]).drop(labels="is_canceled")
spearman = spearman.reindex(spearman.abs().sort_values(ascending=False).index)
spearman = spearman[spearman.abs() > threshold]
spearman

* The 5 features from the Spearman correlation are `lead_time`, `previous_cancellations`, `total_of_special_requests`, `required_car_parking_spaces` and `previous_bookings_not_canceled`

In [ ]:
pps_col = pps_plot_df.set_index("x")["ppscore"].sort_values(ascending=False)
pps_col

* The 7 features from the pps analysis are `deposit_type`, `country`, `agent`, `adr`, `lead_time`, `market_segment` and `previous_cancellations`

* Combine the selected features into one dataframe

In [ ]:
all_features = pearson.index.union(spearman.index).union(pps_col.index)

ranked_df = pd.DataFrame(index=all_features)
ranked_df["pps"] = pps_col
ranked_df["pearson"] = pearson
ranked_df["spearman"] = spearman

# Scores are ordered primarily by pps as the only method able to assess all variables 
ranked_df.sort_values(by="pps", ascending=False)

* Reset the index to include features in the columns list

In [ ]:
ranked_df = ranked_df.reset_index(names="feature")

ranked_df

* Add feature type information 

In [ ]:
feature_type = []

for feature in ranked_df["feature"]:
    if feature in numeric_features:
        feature_type.append("numeric")
    elif feature in categorical_features:
        feature_type.append("categorical")

ranked_df["feature_type"] = feature_type
ranked_df

* Assess agreemement between the 3 methods 

In [ ]:
ranked_df["n_methods"] = ranked_df.count(axis=1, numeric_only=True)
ranked_df

* Rank variables within feature type by agreement between methods and the strongest signal

In [ ]:
ranked_df["strongest_signal"] = ranked_df[["pps", "pearson", "spearman"]].abs().max(axis=1)

ranked_df["rank_in_type"] = (
    ranked_df.sort_values(by=["n_methods", "strongest_signal"], ascending=False)
    .groupby("feature_type")
    .cumcount() + 1
)

ranked_df.sort_values(by=["feature_type", "rank_in_type"])

* Variables are now ranked within their category type, prioritised based on the number of methods that flagged the feature, then the strength of the strongest signal
* The highest ranked categorical feature is `deposit_type`, whereas the highest ranked numerical feature is `lead_time`

---

## Hypothesis Testing

### H1: No deposit bookings cancel more than deposit-secured bookings

* Validation method is chi-squared test on `deposit_type` vs `is_canceled`

* Create crosstab of `deposit_type` and `is_canceled`

In [ ]:
h1_df = pd.crosstab(df["deposit_type"], df["is_canceled"])
h1_df

* No deposit has the largest number of bookings both Not cancelled (74,790) and cancelled (29,649). Bookings with Non Refund deposit type show 14,493 cancelled bookings vs only 93 bookings not cancelled and Refundable deposit is very underrepresented in the dataset with only 162 total bookings, 36 of which went on to cancel 

* Group Non Refund and Refundable into a single "Deposit Secured" category to test H1 (No Deposit vs. deposit-secured)

In [ ]:
h1_binary = df.copy()
h1_binary["deposit_group"] = h1_binary["deposit_type"].map({
    "No Deposit": "No Deposit",
    "Non Refund": "Deposit Secured",
    "Refundable": "Deposit Secured"
})

h1_binary["deposit_group"].value_counts()

* No deposit accounts for the largest group by far with 104,439 rows (~88% of all bookings), the combined NoN Refund and Refundable deposit types account for the remaining 14,748 bookings

* Carry out chi-square test to determine if a statistical relationship exists between `deposit_group` and `is_canceled`

In [ ]:
import pingouin as pg

expected, observed, stats = pg.chi2_independence(data=h1_binary, x="deposit_group", y="is_canceled")
stats


* The Cramer's V effect value of 0.48 shows that there is a statistical relationship between the 2 variables, approaching Cohen's threshold (0.5) for a large effect at this degree of freedom. [*Source: peterstatistics.com*](https://peterstatistics.com/CrashCourse/3-TwoVarUnpair/NomNom/NomNom-2c-Effect-Size.html)

* Assess cancellation rates across the deposit types

In [ ]:
h1_binary_df = pd.crosstab(h1_binary["deposit_group"], h1_binary["is_canceled"], normalize="index")
h1_binary_df.columns = ["not_canceled", "canceled"]

h1_binary_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

* This clearly shows that contrary to the expected outcome stated in H1, 99% of deposit secured bookings cancel compared to only 28% with no deposit

In [ ]:
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt

h1_binary_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.title("Cancellation Rate by Deposit Statua")
plt.ylabel("Cancellation Rate")
plt.xlabel("Deposit Status")
plt.xticks(rotation=0)
plt.legend(labels=["Not Cancelled", "Cancelled"])
plt.show()

* The chart further demonstrates that the relationship is counter to the expected direction with the vast majority of deposit-secured bookings cancelling.

* As seen earlier, the Non Refund deposit type had a far larger volume of bookings than Refundable.
* Plot the deposit types separately to better inform the conslusions

In [ ]:
h1_rate_df = pd.crosstab(df["deposit_type"], df["is_canceled"], normalize="index")

h1_rate_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.title("Cancellation Rate by Deposit Type")
plt.ylabel("Cancellation Rate")
plt.xlabel("Deposit Type")
plt.xticks(rotation=0)
plt.legend(labels=["Not Cancelled", "Cancelled"])
plt.show()

* This confirms that it is the Non Refund deposit type driving the deposit secured result, refundable has a lower cancellation rate than No Deposit

**Conclusion**

The crosstab shows the opposite pattern to that proposed in H1: Non Refund bookings - a deposit-secured type - show a substantially higher cancellation rate than No Deposit bookings.

The chi-square test confirms this relationship is statistically significant (χ² = 27240.71, p < 0.001) with a moderate effect size (Cramer's V = 0.48).

H1 as stated is therefore **rejected** - deposit-secured bookings, specifically the Non Refund type, cancel more than no-deposit bookings.

As a cancellation predictor, the feature remains a risk factor, just not in the way hypothesised.

This may suggest a product misalignment, an overly broad deposit-type categorisation or property specific customer behaviour that the revenue management team can explore further should they wish.

### H2: Bookings with longer lead times have a higher cancellation rate than last-minute bookings

* Validation method is point-biserial correlation between `lead_time` and `is_canceled`

In [ ]:
from scipy import stats

correlation, pvalue = stats.pointbiserialr(df["is_canceled"], df["lead_time"])

h2_stats = pd.DataFrame({
    "correlation": correlation,
    "pvalue": pvalue
}, index=["lead_time_vs_is_canceled"])

h2_stats.style.format("{:.4f}")

* The Pearson's coefficient value of 0.29 shows that there is a statistical relationship between the 2 variables, small but approaching Cohen's threshold (0.3) for a medium effect.
* While Cohen's convention places this result at the upper limit of a 'small' effect, more recent research suggests these thresholds are too strict. Gignac and Szodorai (2016) propose 0.10, 0.20 and 0.30 as more representative benchmarks for relatively small, typical, and relatively large effects respectively — placing this result at the upper end of a typical real-world relationship. [*Source: sciencedirect.com*](https://www.sciencedirect.com/science/article/abs/pii/S0191886916308194)

* Assess cancellation rates by time-bucketed lead times. Last Minute: 0-7 days, Short Range: 8-30 days, Mid Range: 31-90 days and Long Range: 90+ days

In [ ]:
bins = [-np.inf, 7 ,30 ,90, np.inf]
lead_time = pd.cut(df["lead_time"], bins, labels=["Last Minute", "Short Range", "Mid Range", "Long Range"])
h2_df = df.copy()
h2_df["lead_time"] = lead_time
h2_df = pd.crosstab(h2_df["lead_time"], h2_df["is_canceled"], normalize="index")
h2_df.columns = ["not_canceled", "canceled"]
h2_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

* Visualise the cancellation rates by lead time buckets

In [ ]:
h2_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.title("Cancellation Rate by Lead Time Bucket")
plt.ylabel("Cancellation Rate")
plt.xlabel("Lead Time")
plt.legend(labels=["Not Cancelled", "Cancelled"])
plt.xticks(rotation=0)
plt.show()

* The chart clearly displays the relationship, the longer the lead time, the more likely the booking is to cancel

**Conclusion**

The crosstab confirms the pattern proposed in H2: As lead time increases, the proportion of cancelled bookings also increases.

The point-biserial test confirms this relationship is statistically significant (r = 0.29, p < .001) — a small effect by Cohen's convention, though at the upper end of a typical real-world relationship per Gignac and Szodorai (2016).

H2 as stated is therefore **accepted** - longer range bookings cancel more than last-minute.

As a cancellation predictor, the feature is a risk factor.

### H3: Bookings made through the Online TA market segment have a higher cancellation rate than Direct bookings

* Validation method is chi-squared test on `market_segment` "Direct" and "Online TA" vs `is_canceled`

* Create crosstab of cancellation rates of "Online TA" and "Direct" market segments 

In [ ]:
features = ["Online TA", "Direct"]
ota_direct = df[df["market_segment"].isin(features)]
ota_direct

h3_df = pd.crosstab(ota_direct["market_segment"], ota_direct["is_canceled"], normalize="index")
h3_df.columns = ["not_canceled", "canceled"]

h3_df.style.format({"not_canceled": "{:.0%}", "canceled": "{:.0%}"})

* Online TA has a cancellation rate of 37% which is on par with the overall cancellation rate for the dataset, but is more than double that of "Direct" at only 15%

* Carry out chi-square test to determine if a statistical relationship exists between `market_segment` and `is_canceled`

In [ ]:
expected, observed, stats = pg.chi2_independence(data=ota_direct, x="market_segment", y="is_canceled")
stats

* The Cramer's V effect value of 0.18 shows that there is a statistical relationship between the 2 variables, though only a small effect at this degree of freedom as per Cohen's convention. [*Source: peterstatistics.com*](https://peterstatistics.com/CrashCourse/3-TwoVarUnpair/NomNom/NomNom-2c-Effect-Size.html)

In [ ]:
h3_df.plot(kind="bar").yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.axhline(y=df["is_canceled"].mean(), linestyle="--", color="black", label="Overall cancellation rate")
plt.legend(labels=["Overall Cancellation Rate", "Not Cancelled", "Cancelled"])
plt.title("Cancellation Rate Direct and Online TA")
plt.ylabel("Cancellation Rate")
plt.xlabel("Market Segment")
plt.xticks(rotation=0)
plt.show()

* The chart shows that the cancellation rate for Online TA is roughly the same as the overall cancellation rate, while the cancellation rate for direct is considerably below average.

* Assess the booking volume of the 2 market segments

In [ ]:
sns.countplot(data=ota_direct, x="market_segment", hue="is_canceled")
plt.title("Number of bookings by market segment")
plt.ylabel("Number of bookings")
plt.xlabel("Market Segment")
plt.legend(labels=["Not Cancelled", "Cancelled"])
plt.show()

* This chart shows that while OTA bookings cancel at a rate roughly on par with the overall average, their much larger booking volume means they account for a disproportionately high number of cancelled reservations

**Conclusion**

The crosstab confirms the pattern proposed in H3: Online TA bookings have a higher cancellation rate than Direct bookings.

The chi-square test confirms this relationship is statistically significant (χ² = 2146.33, p < 0.001) with a small effect size (Cramer's V = 0.18).

H3 as stated is therefore **accepted** - Online TA bookings have a higher cancellation rate than Direct bookings.

As a predictive feature, market segment (Online TA vs Direct) remains a useful indicator of cancellation risk.

Although the Online TA aligns with the dataset average, bookings from this segment represent a much larger booking volume. Consequently, they contribute a disproportionately large share of all cancelled reservations, which helps explain why they are often perceived operationally as a major source of cancellations.

* Create table of results

In [ ]:
hypothesis_results = pd.DataFrame({
    "hypothesis": ["H1", "H2", "H3"],
    "test": ["Chi-square", "Point-biserial", "Chi-Square"],
    "statistic": ["χ² = 27240.71", "r_pb = 0.29", "χ² = 2146.33"],
    "p_value": ["p < 0.001", "p < 0.001", "p < 0.001"],
    "effect_size": ["Cramer's V = 0.48", "r_pb = 0.29", "Cramer's V = 0.18"],
    "outcome": ["Rejected", "Accepted", "Accepted"]
})

hypothesis_results

* This table clarifies the results of the tests into one clear resource

---

## BR1 Conclusions

This notebook, alongside the exploratory work in the Cancellation EDA, addresses BR1 by formally identifying and validating cancellation risk factors across TCS Hotels' two Portuguese properties.

Three confirmed risk factors emerge from hypothesis testing:

* **Deposit type** is identified in this notebook as the strongest risk factor, though the direction runs counter to initial expectations. Non Refund bookings — not No Deposit bookings — show a substantially higher cancellation rate.
* **Lead time** is a confirmed risk factor: cancellation likelihood rises consistently as lead time increases.
* **Market segment** (Online TA vs Direct) is a confirmed, smaller-effect risk factor. While its cancellation rate sits close to the property-wide average, the segment's disproportionate share of total bookings means it accounts for a large volume of absolute cancellations — a factor operationally significant even where the underlying rate is unremarkable.

Risk Factors from cancellation EDA:

* **Previous Cancellations** was identified in the [cancellation EDA](/jupyter_notebooks/02_cancellation_eda.ipynb) as the strongest visual relationship with a 58 point gap between the cancellation rates of bookings with at least one previous cancellation (92%) and those who have never cancelled before (34%) and was identified by all 3 correlation methods performed as the 2nd strongest numeric predictor after lead time under Spearman correlation
* **Country** was also identified in the [cancellation EDA](/jupyter_notebooks/02_cancellation_eda.ipynb) as a possible predictor - especially when grouped international vs domestic - and was separately surfaced by pps score as the 2nd strongest categorical predictor after deposit type.

Together, these findings indicate that cancellation risk at TCS Hotels is driven less by *when* a guest books relative to arrival alone, and more by the *type of booking commitment* involved — deposit terms, country and booking channel carry stronger signal than volume or seasonality alone. This reframes the client's risk factors away from a simple "early bookers are safer" assumption toward booking-product characteristics as the primary driver of cancellation behaviour.

---

## Feature Engineering Recommendations

**Guest Composition**
* Investigate whether combining `adults`, `children` and `babies` into one total guest count provides a stronger predictive signal
* Assess whether an `is_family` binary flag is of more predictive value than guest counts alone

**Stay Characteristics**
* Evaluate whether combining `stays_in_week_nights` and `stays_in_weekend_nights` into a total length of stay (LOS) improves predictive performance
* Investigate whether arrival day of the week is a factor such as `arrival_is_weekend` or if weekend ratio is an indicator

**Temporal Variables**
* Assess whether cyclical encoding of week number and month better represent seasonality
* Evaluate whether broader seasonal features outperform individual calendar components

**Booking Behaviour**
* Explore whether lead time benefits from transformation or categorisation
* Consider whether previous history can be better represented through derived behavioural features

**Long Tail Compression**
* Investigate whether compiling long-tail categorical features into top-n + other produces a more significant result
* Evaluate the previously discovered binary treatment of `country`: domestic vs international

**Validation**
* Re-run Pearson, Spearman and PPS on all engineered features
* Compare engineered features with the raw variables they replace to inform the decision to keep or revert

---

# Outputs

In [ ]:
import os
try:
  os.makedirs(name='outputs/correlation')
except Exception as e:
  print(e)


* Save RankedFeatures for use in [feature exploration](/jupyter_notebooks/06_feature_exploration.ipynb)
* Save HypothesisResults for possible use in the dashboard

In [ ]:
ranked_df.to_csv("outputs/correlation/RankedFeatures.csv")
hypothesis_results.to_csv("outputs/correlation/HypothesisResults.csv")